# Logistic Regression with Visualizations and Pipeline

This notebook uses the `employee.csv` dataset to build a Logistic Regression classification model.

**Target column:** `income`  
**Goal:** Predict whether income is `<=50K` or `>50K`.

This version includes cleaning, visualizations, preprocessing pipeline, model training, evaluation, prediction, and model saving.

In [ ]:
# Import libraries for data handling
import pandas as pd
import numpy as np

# Import matplotlib for simple visualizations
import matplotlib.pyplot as plt

# Import train_test_split for splitting data
from sklearn.model_selection import train_test_split

# Import preprocessing tools
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Import Logistic Regression model
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics for classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Import pickle to save the trained model
import pickle

## 1. Load the dataset

In [ ]:
# Load dataset
# Keep employee.csv in the same folder as this notebook
df = pd.read_csv("employee.csv")

# Display first few rows
print(df.head())

## 2. Basic dataset understanding

In [ ]:
# Check dataset size
print("Shape:", df.shape)

# Check data types and missing values
print(df.info())

# Check target class count
print(df["income"].value_counts())

## 3. Basic cleaning

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove extra spaces from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Replace '?' with NaN so that missing values are handled clearly
df = df.replace("?", np.nan)

# Drop rows with missing values for this learning notebook
# In real projects, we may use imputation instead of dropping rows
df = df.dropna()

# Check cleaned shape
print("Cleaned shape:", df.shape)

## 4. Simple visualizations

In [ ]:
# Plot income distribution
income_count = df["income"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(income_count.index, income_count.values)
plt.title("Income Class Distribution")
plt.xlabel("Income")
plt.ylabel("Number of Employees")
plt.show()

In [ ]:
# Plot age distribution by histogram
plt.figure(figsize=(8, 5))
plt.hist(df["age"], bins=20)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Employees")
plt.show()

In [ ]:
# Plot education distribution for top 10 education categories
education_count = df["education"].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.bar(education_count.index, education_count.values)
plt.title("Top 10 Education Categories")
plt.xlabel("Education")
plt.ylabel("Number of Employees")
plt.xticks(rotation=45)
plt.show()

## 5. Target selection

In [ ]:
# Select target column
# income is categorical, so it is suitable for Logistic Regression
y = df["income"]

# Select features by removing the target column
X = df.drop("income", axis=1)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## 6. Identify numeric and categorical columns

In [ ]:
# Numeric columns can be scaled
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Categorical columns should be converted to numeric using one-hot encoding
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 7. Train-test split

In [ ]:
# Split data into train and test data
# stratify=y keeps target class ratio similar in train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 8. Build preprocessing and model pipeline

In [ ]:
# Numeric pipeline: scale numeric features
# Scaling helps Logistic Regression work better
numeric_transformer = StandardScaler()

# Categorical pipeline: convert text categories into numeric columns
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Combine numeric and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Create complete pipeline: preprocessing + Logistic Regression
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

print(model)

## 9. Model training

In [ ]:
# Train the complete pipeline
# The pipeline first preprocesses the data and then trains Logistic Regression
model.fit(X_train, y_train)

print("Model training completed")

## 10. Model evaluation

In [ ]:
# Predict classes on test data
y_pred = model.predict(X_test)

# Calculate classification metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=">50K")
recall = recall_score(y_test, y_pred, pos_label=">50K")
f1 = f1_score(y_test, y_pred, pos_label=">50K")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("
Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Display confusion matrix visually
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

## 11. Predict for a new employee

In [ ]:
# Create one new employee record using the same feature columns
new_employee = pd.DataFrame([{
    "age": 35,
    "workclass": "Private",
    "fnlwgt": 200000,
    "education": "Bachelors",
    "educational-num": 13,
    "marital-status": "Married-civ-spouse",
    "occupation": "Exec-managerial",
    "relationship": "Husband",
    "race": "White",
    "gender": "Male",
    "capital-gain": 0,
    "capital-loss": 0,
    "hours-per-week": 40,
    "native-country": "United-States"
}])

# Predict income class for the new employee
prediction = model.predict(new_employee)

# Predict probability for each class
prediction_probability = model.predict_proba(new_employee)

print("Predicted income class:", prediction[0])
print("Class labels:", model.classes_)
print("Prediction probabilities:", prediction_probability)

## 12. Save the model

In [ ]:
# Save the trained pipeline model as a pickle file
# This file can be loaded later for prediction
with open("logistic_regression_employee_model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Model saved as logistic_regression_employee_model.pkl")